### Figure 2-3 C, D, D subpanels, E, Supplemental Figure 2 A, B

In [ ]:

%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.pyplot import cm
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib as mpl
from sklearn.linear_model import LinearRegression
from statsmodels.stats.nonparametric import *
from matplotlib.ticker import MaxNLocator
from paths import DATA_DIR, fig_dir
from spyglass.common import Session

In [ ]:
# Custom schema
from find_my_data import *
from alison_decoding import ClusterlessAcausalResultsSummary
from fig_helpers import *

### figure setup and load data

In [ ]:
set_figure_defaults()

fig_path = fig_dir('figs26')
if not os.path.exists(fig_path):
    os.makedirs(fig_path)

save_fig = False

In [ ]:
# params
position_info_param_name='default_decoding'
remove_hpd_timepoints = True
hpd_percent = 50
hpd_threshold = 50
require_nonlocal_by_segment = False
remove_low_speed_timepoints = True
head_speed_threshold = 10

# data loading info
out_path = f'{DATA_DIR}/big_df_pkls/'
today_now = '20240212'

subject_ids = ['senor', 'chimi', 'j16', 'wilbur', 'peanut']
custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))

In [ ]:
# load behavior and decoding days of data, crosscheck nwbs
big_dfs = {}
for subject_id in subject_ids:
    try:
        big_dfs[subject_id] = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
    except Exception as e:
        print('exception',e)

stable_nwbs = {}
clusterless_nwbs = {}
stable_clusterless_nwbs = {}
for subject_id in subject_ids:
    stable_nwbs[subject_id] = list( (Session & {'session_description LIKE "Spatial bandit task (regular)"'}
                                             & {"subject_id": subject_id}).fetch('nwb_file_name') )
    clusterless_nwbs[subject_id] = list(np.unique((ClusterlessAcausalResultsSummary()
                                                   & spatial_bandit_query_by_rat(rat_list=[subject_id])).fetch('nwb_file_name')))
    if subject_id == 'j16':
        stable_nwbs['j16'].remove('mediumnwb20230802_.nwb')
    if subject_id == 'chimi':
        stable_nwbs['chimi'].remove('chimi20200216_new_.nwb')
    if subject_id == 'senor':
        stable_nwbs['senor'].remove('senor20201030_.nwb')
    stable_clusterless_nwbs[subject_id] = [nwb for nwb in clusterless_nwbs[subject_id] if nwb in stable_nwbs[subject_id]]
print(stable_clusterless_nwbs)

In [ ]:
is_mapped_seg_a_leaf_map = {0:False, 1:True, 2:True, 3:False, 4:True, 5:True, 6:False, 7:True, 8:True}
segs_to_patch_map = {0:1, 1:1, 2:1, 3:2, 4:2, 5:2, 6:3, 7:3, 8:3}
p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]

# Restrict big df to those with clusterless data, and agument for related analyses, ensure no 100all 50all
all_rat_big_dfs_stable = {}
for subject_id in subject_ids:
    df = big_dfs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_clusterless_nwbs[subject_id])]
    df_stable['is_actual_seg_mapped_a_leaf'] = df_stable[['actual_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['is_mental_seg_mapped_a_leaf'] = df_stable[['mental_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['mental_patch_mapped'] = df_stable[['mental_segment_mapped']].applymap(segs_to_patch_map.get)
    all_rat_big_dfs_stable[subject_id] = df_stable[~df_stable[p_rew_cols].eq(df_stable['p_rew_leaf1'], axis=0).all(axis=1)]

In [ ]:
# Filter data to segments of interest 
hpd_percent = 50 # or 95
hpd_thresh_cm = 50 # 50
use_abs_ahbeh_thresh = False
abs_ahbeh_thresh_cm = 10
quantile = .9

big_dfs_firstlast_nonlocal_incljump_grouped = {}
for subject_id in subject_ids:
    big_df = all_rat_big_dfs_stable[subject_id]
    
    # limit to first or last track segment, add any hpd and ahbeh restrictions for quality control
    big_df_firstlast = big_df[np.logical_and(
                                    np.logical_or(big_df['is_first_seg_of_trial']==True,
                                                  big_df['is_last_seg_of_trial']==True),
                                    big_df[f'spatial_coverage_{hpd_percent}_hpd']<hpd_thresh_cm,
                                    )]
    if use_abs_ahbeh_thresh:
        big_df_firstlast = big_df_firstlast[big_df_firstlast['abs_ahead_behind_distance']>=abs_ahbeh_thresh_cm]
    
    big_df_firstlast_nonlocal = big_df_firstlast[big_df_firstlast['nonlocal_by_segment']==True]
    
    # find nonlocal stay and switch consistent content
    big_df_firstlast_nonlocal['is_mental_seg_leaf_in_patch_or_elsewhere'] = np.logical_and(
                                                                    big_df_firstlast_nonlocal['nonlocal_by_patch']==False,
                                                                    big_df_firstlast_nonlocal['is_mental_seg_mapped_a_leaf']==True)
    big_df_firstlast_nonlocal['is_mental_seg_elsewhere_or_leaf_in_patch'] = ~big_df_firstlast_nonlocal['is_mental_seg_leaf_in_patch_or_elsewhere']
    
    # now do all the groupings to calc things for first/final seg data
    # these calcultaions are only during nonlocal by seg times, not full trial time
    big_df_grouped = big_df_firstlast_nonlocal.groupby(
            by=['nwb_file_name', 'epoch_number', 'trial_number_by_epoch', 'stem_switch', 'stem', 'leaf', 'stemchoice', 'reward', 'is_first_seg_of_trial',
                'trials_from_prior_switch', 'trials_from_next_switch',]
        ).apply(
            lambda x_df: pd.Series({
                'prop_elsewhere_vs_neighbor_leaf': len(x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]) / len(x_df),
                'len_elsewhere': len(x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]),
                'len_neighbor_leaf': len(x_df[x_df['is_mental_seg_leaf_in_patch_or_elsewhere']]),
                'ahbeh_mean':x_df['abs_ahead_behind_distance'].mean(),
                'ahbeh_mean_neighbor_leaf':x_df[x_df['is_mental_seg_leaf_in_patch_or_elsewhere']]['abs_ahead_behind_distance'].mean(),
                'ahbeh_mean_elsewhere':x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]['abs_ahead_behind_distance'].mean(),
                'ahbeh_max':x_df['abs_ahead_behind_distance'].max(),
                'ahbeh_max_neighbor_leaf':x_df[x_df['is_mental_seg_leaf_in_patch_or_elsewhere']]['abs_ahead_behind_distance'].max(),
                'ahbeh_max_elsewhere':x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]['abs_ahead_behind_distance'].max(),
                f'ahbeh_quantile{int(100*quantile)}':x_df['abs_ahead_behind_distance'].quantile(q=quantile),
                f'len_nonlocal': len(x_df),
            })
        ).reset_index()
    big_dfs_firstlast_nonlocal_incljump_grouped[subject_id] = big_df_grouped

In [ ]:
# Make all animal concatenated df, reset index to maek subject_id col
concatenated_df = pd.concat(big_dfs_firstlast_nonlocal_incljump_grouped.values(), keys=big_dfs_firstlast_nonlocal_incljump_grouped.keys(), names=['subject_id'])
concatenated_df.reset_index(level=0, inplace=True)
concatenated_df.reset_index(drop=True, inplace=True)

In [ ]:
# check n with agg
group_for_n = concatenated_df.groupby([ 'is_first_seg_of_trial','subject_id','stem_switch'])
agg_content = group_for_n['prop_elsewhere_vs_neighbor_leaf'].agg(['size', 'count']).rename(columns={'size': 'length'})
agg_extent = group_for_n['ahbeh_max'].agg(['size', 'count']).rename(columns={'size': 'length'})


In [ ]:
# contet n
print("Aggregations for content:")
agg_content

In [ ]:
# extent n
print("Aggregations for extent:")
agg_extent

In [ ]:
# how many trials nonlocal per current params
denom = {'j16':10191,
'chimi':8970,
'senor': 5760, #4680,
'wilbur': 7917,
'peanut': 5220}

trial_counts = (
    concatenated_df.drop_duplicates(subset=['subject_id', 'nwb_file_name', 'epoch_number', 'trial_number_by_epoch'])
      .groupby('subject_id')
      .size()
      .reset_index(name='n_trials')
)

trial_counts['denominator'] = trial_counts['subject_id'].map(denom)
trial_counts['prop'] = trial_counts['n_trials'] / trial_counts['denominator']
trial_counts

In [ ]:
# add per-rat baseline subtracted data, baseline excludes switch for content analyses
grouped = concatenated_df[concatenated_df['stem_switch']==False].groupby(['subject_id', 'is_first_seg_of_trial'])

# Calculate the mean baseline for each group - without switches, here!!
mean_baseline = grouped['prop_elsewhere_vs_neighbor_leaf'].transform('mean')
mean_baseline2 = grouped['ahbeh_max'].transform('mean')
mean_baseline3 = grouped['len_elsewhere'].transform('mean')
mean_baseline4 = grouped['len_neighbor_leaf'].transform('mean')
mean_baseline5 = grouped['len_nonlocal'].transform('mean')

# Create the new column
concatenated_df['prop_elsewhere_vs_neighbor_leaf_minus_avg'] = concatenated_df['prop_elsewhere_vs_neighbor_leaf'] - mean_baseline
concatenated_df['ahbeh_max_minus_avg'] = concatenated_df['ahbeh_max'] - mean_baseline2
concatenated_df['len_elsewhere_minus_avg'] = concatenated_df['len_elsewhere'] - mean_baseline3
concatenated_df['len_neighbor_leaf_minus_avg'] = concatenated_df['len_neighbor_leaf'] - mean_baseline4
concatenated_df['len_nonlocal_minus_avg'] = concatenated_df['len_nonlocal'] -mean_baseline5

# Baseline subtracted analyses must only be on stay trials, not switch

#### F2 E, F3 E: stay vs switch content

In [ ]:
# Fig params
figwidth = TWO_COLUMN/3
figheight = figwidth
custom_palette_by_rat = {subject_id: next(custom_colors_by_rat) for subject_id in subject_ids}

In [ ]:
# Plot stay vs switch ahead and behind for each rat and all rats
withline=True
legendon=False
metrics = ['prop_elsewhere_vs_neighbor_leaf']
ci = 95
dodge=True
customylim = True
scale = .8
markersize=.1

for metric in metrics:
    for is_first_seg_of_trial in [True,False]:
        concatenated_df_copy = concatenated_df.copy() # don't modify orig
        d = concatenated_df_copy[concatenated_df_copy['is_first_seg_of_trial']==is_first_seg_of_trial] # get first/final seg
        d['stem_switch_float'] = d['stem_switch'].astype(float)
        plt.figure(figsize=(figwidth/2,figheight))

        # data points with ci
        if withline:
            sns.lineplot(data = d, x='stem_switch_float', y=metric, ci=0, hue='subject_id',  palette=custom_palette_by_rat,lw=.3, label=None, markersize=markersize)
        sns.pointplot(data = d, x='stem_switch', y=metric, scale=scale,ci=ci, hue='subject_id', label=f'Rat {subject_id[0].upper()}', dodge=dodge, palette=custom_palette_by_rat, join=False,) #markersize=markersize) #markersize removed for version compatability for export

        # label
        handles, labels = plt.gca().get_legend_handles_labels()
        plt.legend(frameon=False, bbox_to_anchor=(1.01,1), loc='upper left', handles=handles, labels=[f'Rat {label[0].upper()}' for label in labels])
        plt.title(f'First seg: {is_first_seg_of_trial}, ci: {ci}', y=1.01, fontsize=4)
        plt.xlabel('')
        if metric == 'prop_elsewhere_vs_neighbor_leaf':
            plt.ylabel(f'Proportion of non-local activity\nalong Switch paths')
        elif metric == 'ahbeh_max':
            plt.ylabel(f'Max Ahead-Behind Distance')
        plt.xticks([0,1],['Stay\nTrial','Switch\nTrial'])
        if customylim:
            if metric[0:4]=='prop':
                plt.ylim(0,1)
            else:
                plt.ylim(0,100)
        sns.despine()

        # save
        if save_fig:
            fig_name = f'stay_vs_switch_colors_{metric}_firstseg{is_first_seg_of_trial}_ci{ci}_dodge{dodge}_customylim{customylim}_scale{scale}_withline{withline}_ahbehthresh{use_abs_ahbeh_thresh}_ahbehgt{abs_ahbeh_thresh_cm}'
            plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5)   
        
        # compare stay vs switch stats per rat
        for subject_id in subject_ids:
            d_rat = d[d.subject_id == subject_id]
            print(f'\nWILCOXON RANK SUM {metric}, {subject_id}, first_seg: {is_first_seg_of_trial}, Stay vs Switch\n')
            x1 = d_rat[d_rat['stem_switch_float']==1][metric]
            x2 = d_rat[d_rat['stem_switch_float']==0][metric]
            result = rank_compare_2indep(x1, x2, use_t=True)
            print(result.summary())
            print(result)
            print(f'confint: {result.conf_int()}')
            
        plt.show()

#### F2 C,D, F3 C,D, SF2 A,B peri-switch content

In [ ]:
subject_ids_and_all = subject_ids + ['allrats']

# params
figwidth = TWO_COLUMN/3
figheight = figwidth
custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))
custom_palette_by_rat = {subject_id: next(custom_colors_by_rat) for subject_id in subject_ids}
all_rat_color = 'black'
xmax = 20 # trials
trial_subsets = ['trials_from_next_switch', 'trials_from_prior_switch']
include_switches=False
err_style = 'bars'
markersize=5
ci=95
linewidth=.5
show_shuffle = False
show_linreg=True
offset = 5
metrics = ['prop_elsewhere_vs_neighbor_leaf',] # 'ahbeh_max'] # not doing extent while incl lin regs


In [ ]:
for is_first_seg_of_trial in [True,False]:
    for metric in metrics:
        for subject_id in subject_ids_and_all: # set up
            if subject_id == 'allrats':
                metric = f'{metric}_minus_avg'

            fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(figwidth,figheight), sharey=True)

            for i,trial_subset in enumerate(trial_subsets):
                concatenated_df_copy = concatenated_df.copy()
                if subject_id != 'allrats':
                    data_df = concatenated_df_copy[concatenated_df_copy['subject_id']==subject_id] #concatenated_df        
                else:
                    data_df = concatenated_df_copy
                    
                data_df = data_df[data_df['is_first_seg_of_trial']==is_first_seg_of_trial]
                
                # Filter data to only what to plot
                if (trial_subset == 'trials_from_next_switch') & (is_first_seg_of_trial == False):
                    data_df = data_df[(data_df[trial_subset]<=-1) & (data_df[trial_subset]>=-xmax)]
                elif (trial_subset == 'trials_from_next_switch') & (is_first_seg_of_trial == True):
                    data_df = data_df[(data_df[trial_subset]<=0) & (data_df[trial_subset]>=-xmax)]
                elif (trial_subset == 'trials_from_prior_switch') & (is_first_seg_of_trial == False):
                    data_df = data_df[(data_df[trial_subset]>=0) & (data_df[trial_subset]<=xmax)]
                elif (trial_subset == 'trials_from_prior_switch') & (is_first_seg_of_trial == True):
                    data_df = data_df[(data_df[trial_subset]>=1) & (data_df[trial_subset]<=xmax)]

                if include_switches == False:
                    data_df = data_df[data_df['stem_switch']==False]

            # Plot content or extent variables
            # first plot for CI alpha, then plot bigger markers on top of the data
                sns.lineplot(data=data_df,
                             x=trial_subset, y=metric,
                             markersize=markersize, marker='o', err_style = err_style,
                             color=custom_palette_by_rat[subject_id] if subject_id in subject_ids else all_rat_color,
                             ax=axes[i], ci=ci,  mew=0, alpha=.3,
                             err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                             zorder=99,linewidth=0)
                sns.lineplot(data=data_df,
                             x=trial_subset, y=metric,
                             markersize=markersize, marker='o', err_style = err_style,
                             color=custom_palette_by_rat[subject_id] if subject_id in subject_ids else all_rat_color,
                             ax=axes[i], errorbar=None,  mew=0,
                            #  ax=axes[i], ci=0,  mew=0, # for export edits compatability
                             err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                             zorder=100,linewidth=0)

                if show_linreg:
                    x = np.array(data_df[trial_subset]).reshape((-1,1))
                    y = np.array(data_df[metric]) 
                    model = LinearRegression().fit(x,y)
                    x_fit = np.linspace(x.min(), x.max(), 100).reshape(-1,1)
                    y_fit = model.predict(x_fit)
                    intercept = model.intercept_
                    slope = model.coef_[0]
                    axes[i].plot(x_fit, y_fit, color='black')
                    # statsmodels to get p vals and slopes and to fit
                    X = sm.add_constant(x) # add col of ones to include an intercept in model
                    model = sm.OLS(y, X, hasconst=True)
                    results = model.fit()
                    sm_slope = results.params[1]
                    sm_p_value = results.pvalues[1]
                    print(f'sklearn slope: {slope}, Slope: {sm_slope}, Pval: {sm_p_value}')
                if metric[0:4] == 'prop':
                    if subject_id != 'allrats':
                        axes[i].set_ylim(0,.6)
                        axes[0].set_ylabel(f'Proportion of Non-local activity\nalong Switch paths')
                    else:
                        axes[i].set_ylim(-.3,.3)
                        axes[0].set_ylabel(f'Baseline-subtracted\nProprtion of Non-local activity\nalong Switch paths')
                elif metric[0:4] == 'ahbe':
                    if subject_id != 'allrats':
                        if is_first_seg_of_trial:
                            axes[i].set_ylim(0,120)
                        else:
                            axes[i].set_ylim(0,80)
                        axes[0].set_ylabel(f'Max. Non-local Distance')
                    else:
                        if is_first_seg_of_trial:
                            axes[i].set_ylim(-60,60)
                        else:
                            axes[i].set_ylim(-40,40)
                        axes[0].set_ylabel(f'Baseline-subtracted\nMax. Non-local Distance')
                elif metric[0:6] =='len_el':
                    if subject_id != 'allrats':
                        if is_first_seg_of_trial:
                            axes[i].set_ylim(-0,110)
                        else:
                            axes[i].set_ylim(-0,110)
                        axes[0].set_ylabel(f'Switch path duration (bins)')
                    else:
                        if is_first_seg_of_trial:
                            axes[i].set_ylim(-30,60)
                        else:
                            axes[i].set_ylim(-30,60)
                        axes[0].set_ylabel(f'Baseline-subtracted Switch\npath duration (bins)')
                elif metric[0:6] == 'len_ne':
                    if subject_id != 'allrats':
                        if is_first_seg_of_trial:
                            axes[i].set_ylim(-0,60)
                        else:
                            axes[i].set_ylim(-0,60)
                        axes[0].set_ylabel(f'Stay path duration (bins)')
                    else:
                        if is_first_seg_of_trial:
                            axes[i].set_ylim(-30,60)
                        else:
                            axes[i].set_ylim(-30,60)
                        axes[0].set_ylabel(f'Baseline-subtracted Stay\npath duration (bins)')
                axes[i].set_xlabel('')
            sns.despine(offset=offset)
            axes[0].set_xlim(-xmax-1,0)
            axes[1].set_xlim(0,xmax+1)
            axes[0].set_xticks([-xmax,-10,-1], )
            axes[1].set_xticks([1,10,xmax],)
            axes[0].spines['bottom'].set_bounds(-xmax,-1)
            axes[1].spines['bottom'].set_bounds(1,xmax)    
            axes[1].yaxis.set_visible(False)
            axes[1].spines['left'].set_visible(False)

            plt.suptitle(f'Rat {subject_id[0].upper()}, first seg: {is_first_seg_of_trial}', fontsize=6) #, y=1.01)
            fig.text(0.5,-.07, 'Trials since Switch Trial',ha='center')
            fig.subplots_adjust(wspace=.1)

            if save_fig:
                fig_name = f'stays_periswitch_{subject_id}_{metric}_firstseg{is_first_seg_of_trial}_inclswitch{include_switches}_err{err_style}_ci{ci}_shuff{show_shuffle}_linreg{show_linreg}_xmax{xmax}_w{figwidth}_h{figheight}_concatc{all_rat_color}_s{markersize}_lw{linewidth}_offset{offset}'
                plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5)     

            plt.show()

In [ ]:
# lin reg full stats
pd.reset_option('display.precision')
pd.reset_option('display.float_format')
pd.set_option('display.float_format', lambda x: f'{x:.8g}')

for is_first_seg_of_trial in [True,False]:
    for metric in metrics:
        for subject_id in subject_ids_and_all:
            if subject_id == 'allrats':
                metric = f'{metric}_minus_avg'

            for i,trial_subset in enumerate(trial_subsets):
                concatenated_df_copy = concatenated_df.copy()
                if subject_id != 'allrats':
                    data_df = concatenated_df_copy[concatenated_df_copy['subject_id']==subject_id]        
                else:
                    data_df = concatenated_df_copy
                    
                data_df = data_df[data_df['is_first_seg_of_trial']==is_first_seg_of_trial]
                
                # Filter data to only what to plot
                if (trial_subset == 'trials_from_next_switch') & (is_first_seg_of_trial == False):
                    data_df = data_df[(data_df[trial_subset]<=-1) & (data_df[trial_subset]>=-xmax)]
                elif (trial_subset == 'trials_from_next_switch') & (is_first_seg_of_trial == True):
                    data_df = data_df[(data_df[trial_subset]<=0) & (data_df[trial_subset]>=-xmax)]
                elif (trial_subset == 'trials_from_prior_switch') & (is_first_seg_of_trial == False):
                    data_df = data_df[(data_df[trial_subset]>=0) & (data_df[trial_subset]<=xmax)]
                elif (trial_subset == 'trials_from_prior_switch') & (is_first_seg_of_trial == True):
                    data_df = data_df[(data_df[trial_subset]>=1) & (data_df[trial_subset]<=xmax)]

                if include_switches == False:
                    data_df = data_df[data_df['stem_switch']==False]
                    
                print(f'\n\nMETRIC {metric}, TRIAL_SUBSET {trial_subset}, FIRST_SEG {is_first_seg_of_trial}')
                if subject_id == 'allrats':
                    # now run model
                    print(f'Now working on mixed model for {subject_id.upper()} {metric} {trial_subset} {is_first_seg_of_trial}')
                    model = smf.mixedlm(f"{metric} ~ {trial_subset}", data=data_df, groups=data_df['subject_id'])
                    result=model.fit()
                    print(result.summary())
                    print('pvals: ', result.pvalues)
                else:
                    print(f'Now working on lin reg for {subject_id.upper()} {metric} {trial_subset} {is_first_seg_of_trial}')
                    x = np.array(data_df[trial_subset]).reshape((-1,1))
                    y = np.array(data_df[metric]) # this is on full distribution not the means here
                    x_with_const = sm.add_constant(x) # add constant for stats models fmting
                    model = sm.OLS(y, x_with_const)
                    result=model.fit()
                    print(result.summary())
                    print('pvals: ', result.pvalues, '\n\n')

#### F2 D, F3 D, SF2 A,B shuffles

In [ ]:
# Functions

# shuffling code
def shifting_fxn(rat_df):
    len_x_df = len(rat_df)
    shift_by_t = np.random.randint(0,len_x_df)
    for metric in metrics:
        rat_df[f'{metric}_shifted'] = np.roll(rat_df[metric], shift_by_t)
    return rat_df

def get_shuff_and_true_slopes(shuff_data, data_df, metrics, trial_labels, is_first_seg_of_trial, include_switches):
#     shuff_data = first_seg_shuffle_summary # first_seg_shuffle_summary_dict[subject_id], or change to final_seg prefix 
#     data_df = concatenated_df[concatenated_df['subject_id'] == 'j16'] # the original data the shuff summary was run on to find the true slope of means

    shuff_slopes = {}
    true_slopes = {}
    for trial_label in trial_labels:
        shuff_slopes[trial_label] = {}
        true_slopes[trial_label] = {}
        if trial_label == 'trials_from_prior_switch':
            xmin = 1
            xmax = 20
        elif trial_label == 'trials_from_next_switch':
            xmin = -20
            xmax = -1
        else:
            print('Do not recognize a trial label!!')
        for metric in metrics:
            shuff_slopes[trial_label][metric] = []
            for shuff_df in shuff_data[trial_label]: # go through n_shifts times
#                 print(shuff_df.shape)
                shuff_df_filtered = shuff_df[(shuff_df[trial_label] >= xmin) & (shuff_df[trial_label] <= xmax)]
                x = np.array(shuff_df_filtered[trial_label]).reshape((-1,1))
                y = np.array(shuff_df_filtered[f'{metric}_shifted'])
                #print(x.shape, y.shape)
                model = LinearRegression().fit(x,y)
    #             r_sq = model.score(x,y)
    #             intercept = model.intercept_
                slope = model.coef_[0]
                shuff_slopes[trial_label][metric].append(slope) # keep track of slope on each shuffle
                #print(slope, intercept, r_sq)
            true_slopes[trial_label][metric] = []
            if is_first_seg_of_trial is not None:
                reg_data = data_df[(data_df['is_first_seg_of_trial']==is_first_seg_of_trial)]
            else:
                reg_data = data_df
            if include_switches == False:
                reg_data = reg_data[reg_data['stem_switch']==False]
            reg_data = reg_data[(reg_data[trial_label]<=xmax) & (reg_data[trial_label]>=xmin)]             
            means = reg_data.groupby(by=[trial_label])[metric].mean().reset_index(name=f'{metric}_mean')
            x = np.array(means[trial_label]).reshape((-1,1))
            y = np.array(means[f'{metric}_mean'])
    #         x, y
            model = LinearRegression().fit(x,y)
    #         r_sq = model.score(x,y)
    #         intercept = model.intercept_
            true_slope = model.coef_[0]
            true_slopes[trial_label][metric].append(true_slope)
    return shuff_slopes, true_slopes



def plot_shuff_and_true_slopes_formatted(shuff_slopes, true_slopes, trial_labels, metrics, confidence_level, is_first_seg_of_trial, include_switches, subject_id, n_shifts,save_fig = False,offset=5,despine=True,legendon=True,titleon=True):
    lower_p = ((1.0 - confidence_level) / 2.0) * 100
    upper_p = (confidence_level + ((1.0 - confidence_level) / 2.0)) * 100
    # bins = np.arange(np.min(shuff_slopes[trial_label][metric])-2*step, np.max(shuff_slopes[trial_label][metric])+2*step, step)
    for trial_label in trial_labels:
        if trial_label == 'trials_from_prior_switch':
            xmin = 1
            xmax = 20
        elif trial_label == 'trials_from_next_switch':
            xmin = -20
            xmax = -1
        for metric in metrics:
            plt.figure(figsize=(TWO_COLUMN/4,(TWO_COLUMN/4)*GOLDEN_RATIO))
            counts, bins, patches = plt.hist(shuff_slopes[trial_label][metric], bins=40, density=True, color='silver', histtype='stepfilled', label="Shuffled Slopes")
            plt.axvline(true_slopes[trial_label][metric], color='black', label='Observed Slope', linewidth=2)
            # also plot 95 or 99 % ci
            max_count = max(counts)
            bin_width = np.mean(np.diff(bins))
            text_x_offset = bin_width*0.7
            lower_ci = np.percentile(shuff_slopes[trial_label][metric], lower_p)
            upper_ci = np.percentile(shuff_slopes[trial_label][metric], upper_p)
            plt.axvline(upper_ci, color='grey', linestyle='--',label=f'_{(confidence_level*100)}% Confidence Bounds')
            plt.text(lower_ci+text_x_offset, max_count, "2.5% CI", rotation=90, verticalalignment='top',horizontalalignment='left',color='grey')
            plt.text(upper_ci, max_count, "97.5% CI", rotation=90, verticalalignment='top',horizontalalignment='right',color='grey')
            plt.axvline(lower_ci, color='grey', linestyle='--')
            plt.xlabel('Linear Regression Slope')
            plt.ylabel('Density')
            if legendon:
                plt.legend(frameon=False, bbox_to_anchor=(1.01, .8), loc='upper left')
            if titleon:
                plt.title(f'{subject_id}, first seg: {is_first_seg_of_trial}, metric: {metric},\nincl_switch: {include_switches}, x: {trial_label},\nxmin: {xmin}, xmax: {xmax}, ci: {confidence_level}, n_shifts:{n_shifts}', fontsize=4)
            if despine==True:
                sns.despine(offset=offset)
            ax = plt.gca()
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
            if save_fig:
                fig_name = f'lin_reg_coef_formatted_{subject_id}_{is_first_seg_of_trial}_{trial_label}_{metric}_includeswitch{include_switches}_xmin{xmin}_xmax{xmax}_ci{confidence_level}_nshifts{n_shifts}_despine{despine}_offset{offset}_legend{legendon}_title{titleon}'
                save_figure(fig_path, fig_name)  
            plt.show()
            
# Function to calculate p-value
def calculate_p_value(observed, shuffled):
    shuffled = np.array(shuffled)
    n = len(shuffled)
    greater_count = np.sum(shuffled >= observed)
    lesser_count = np.sum(shuffled <= observed)
    # Calculate two-tailed p-value
    p_value = 2 * min(greater_count, lesser_count) / n
    # Ensure the p-value is never zero by using the smallest possible non-zero p-value
    if p_value == 0:
        p_value = 2 / n
    return p_value

def compare_shuff_true_slopes_p(shuff_slopes, true_slopes):
    # Dictionary to store p-values
    p_value_dict = {}

    # Iterate over categories and metrics to calculate p-values
    for category in shuff_slopes:
        p_value_dict[category] = {}
        for metric in shuff_slopes[category]:
            observed_slope = true_slopes[category][metric][0]
            shuffled_slopes = shuff_slopes[category][metric]
            p_value = calculate_p_value(observed_slope, shuffled_slopes)
            p_value_dict[category][metric] = p_value

    # Display the p-value dictionary
    print(f"P-Value Dictionary\n", p_value_dict)
    return p_value_dict

def get_shuff_means(df, n_shifts, is_first_seg_of_trial, metrics, trial_labels, confidence_level=.999, include_switches=False):
    if is_first_seg_of_trial is not None:
        df = df[df['is_first_seg_of_trial']==is_first_seg_of_trial]
    if include_switches==False:
        df = df[df['stem_switch']==False]
    grouped_df_lists = {trial_label:[] for trial_label in trial_labels}
    
    for n in range(n_shifts):
        df = df.groupby(by=['subject_id']).apply(shifting_fxn) #.reset_index()
        for i in grouped_df_lists.keys():
            df_out = df.groupby(by=[i]).apply(lambda x_df: x_df[[f'{metric}_shifted' for metric in metrics]].mean()).reset_index()
            grouped_df_lists[i].append(df_out)
    
    return grouped_df_lists

In [ ]:
# PERFORM SHUFFLES

# FIRST SEG
is_first_seg_of_trial = True

metrics = ["prop_elsewhere_vs_neighbor_leaf_minus_avg", "ahbeh_max_minus_avg"] # ALWAYS PROP FIRST AND AHBEH SECOND
trial_labels = ['trials_from_prior_switch','trials_from_next_switch']
n_shifts = 1000
confidence_level = .95
include_switches=False # particularly for content analyses

first_seg_shuffle_summary = get_shuff_means(concatenated_df,
                                                n_shifts=n_shifts,
                                                is_first_seg_of_trial=is_first_seg_of_trial,
                                                metrics=metrics,
                                                trial_labels=trial_labels,
                                                confidence_level=confidence_level,
                                                include_switches=include_switches)

metrics = ["prop_elsewhere_vs_neighbor_leaf", "ahbeh_max"] # ALWAYS PROP FIRST AND AHBEH SECOND
first_seg_shuffle_summary_dict = {}
for subject_id in subject_ids:   
    first_seg_shuffle_summary_dict[subject_id] = get_shuff_means(concatenated_df[concatenated_df['subject_id'] == subject_id],
                                                n_shifts=n_shifts,
                                                is_first_seg_of_trial=is_first_seg_of_trial,
                                                metrics=metrics,
                                                trial_labels=trial_labels,
                                                confidence_level=confidence_level,
                                                include_switches=include_switches)
    
# REPEAT for FINAL SEG
is_first_seg_of_trial = False

metrics = ["prop_elsewhere_vs_neighbor_leaf_minus_avg", "ahbeh_max_minus_avg"] # ALWAYS PROP FIRST AND AHBEH SECOND
final_seg_shuffle_summary = get_shuff_means(concatenated_df,
                                                n_shifts=n_shifts,
                                                is_first_seg_of_trial=is_first_seg_of_trial,
                                                metrics=metrics,
                                                trial_labels=trial_labels,
                                                confidence_level=confidence_level,
                                                include_switches=include_switches)

metrics = ["prop_elsewhere_vs_neighbor_leaf", "ahbeh_max"] # ALWAYS PROP FIRST AND AHBEH SECOND
final_seg_shuffle_summary_dict = {}
for subject_id in subject_ids:   
    final_seg_shuffle_summary_dict[subject_id] = get_shuff_means(concatenated_df[concatenated_df['subject_id'] == subject_id],
                                                n_shifts=n_shifts,
                                                is_first_seg_of_trial=is_first_seg_of_trial,
                                                metrics=metrics,
                                                trial_labels=trial_labels,
                                                confidence_level=confidence_level,
                                                include_switches=include_switches)

In [ ]:
# ALL RATS next steps

# FIRST SEG
subject_id = 'all_rats'
metrics = ["prop_elsewhere_vs_neighbor_leaf_minus_avg", "ahbeh_max_minus_avg"] # ALWAYS PROP FIRST AND AHBEH SECOND
trial_labels = ['trials_from_prior_switch','trials_from_next_switch']

# first seg
is_first_seg_of_trial = True

shuff_data = first_seg_shuffle_summary
data_df = concatenated_df

legendon=False
titleon=False
despine=True

shuff_slopes, true_slopes = get_shuff_and_true_slopes(shuff_data=shuff_data,
                                                      data_df=data_df,
                                                      metrics=metrics,
                                                      trial_labels=trial_labels,
                                                      is_first_seg_of_trial=is_first_seg_of_trial,
                                                     include_switches=include_switches,)

plot_shuff_and_true_slopes_formatted(shuff_slopes=shuff_slopes,
                          true_slopes=true_slopes,
                          metrics=metrics,
                          trial_labels=trial_labels,
                          is_first_seg_of_trial=is_first_seg_of_trial,
                          confidence_level=confidence_level,
                          include_switches=include_switches,
                           subject_id=subject_id,
                           n_shifts=n_shifts,
                          save_fig=save_fig,
                             offset=offset,
                             despine=despine,
                                    legendon=legendon,
                                    titleon=titleon)
print(subject_id)
print('is_first_seg_of_trial: ', is_first_seg_of_trial)
compare_shuff_true_slopes_p(shuff_slopes, true_slopes)

In [ ]:
# ALL RATS final seg
is_first_seg_of_trial = False

shuff_data = final_seg_shuffle_summary
data_df = concatenated_df

shuff_slopes, true_slopes = get_shuff_and_true_slopes(shuff_data=shuff_data,
                                                      data_df=data_df,
                                                      metrics=metrics,
                                                      trial_labels=trial_labels,
                                                      is_first_seg_of_trial=is_first_seg_of_trial,
                                                     include_switches=include_switches,)
plot_shuff_and_true_slopes_formatted(shuff_slopes=shuff_slopes,
                          true_slopes=true_slopes,
                          metrics=metrics,
                          trial_labels=trial_labels,
                          is_first_seg_of_trial=is_first_seg_of_trial,
                          confidence_level=confidence_level,
                          include_switches=include_switches,
                           subject_id=subject_id,
                           n_shifts=n_shifts,
                          save_fig=save_fig,
                             offset=offset,
                             despine=despine,
                                    legendon=legendon,
                                    titleon=titleon)
print(subject_id)
print('is_first_seg_of_trial: ', is_first_seg_of_trial)
compare_shuff_true_slopes_p(shuff_slopes, true_slopes)

In [ ]:
# Each rat first then final 
metrics = ["prop_elsewhere_vs_neighbor_leaf", "ahbeh_max"] # ALWAYS PROP FIRST AND AHBEH SECOND

for subject_id in subject_ids:
    # first seg
    is_first_seg_of_trial = True

    shuff_data = first_seg_shuffle_summary_dict[subject_id]
    data_df = concatenated_df[concatenated_df['subject_id'] == subject_id]

    shuff_slopes, true_slopes = get_shuff_and_true_slopes(shuff_data=shuff_data,
                                                          data_df=data_df,
                                                          metrics=metrics,
                                                          trial_labels=trial_labels,
                                                          is_first_seg_of_trial=is_first_seg_of_trial,
                                                         include_switches=include_switches,)
    plot_shuff_and_true_slopes_formatted(shuff_slopes=shuff_slopes,
                              true_slopes=true_slopes,
                              metrics=metrics,
                              trial_labels=trial_labels,
                              is_first_seg_of_trial=is_first_seg_of_trial,
                              confidence_level=confidence_level,
                              include_switches=include_switches,
                            subject_id = subject_id,
                           n_shifts=n_shifts,
                              save_fig=save_fig,
                             offset=offset,
                             despine=despine,
                                    legendon=legendon,
                                    titleon=titleon)
    print(subject_id)
    print('is_first_seg_of_trial: ', is_first_seg_of_trial)
    compare_shuff_true_slopes_p(shuff_slopes, true_slopes)
    
    # final seg
    is_first_seg_of_trial = False

    shuff_data = final_seg_shuffle_summary_dict[subject_id]
    data_df = concatenated_df[concatenated_df['subject_id'] == subject_id]

    shuff_slopes, true_slopes = get_shuff_and_true_slopes(shuff_data=shuff_data,
                                                          data_df=data_df,
                                                          metrics=metrics,
                                                          trial_labels=trial_labels,
                                                          is_first_seg_of_trial=is_first_seg_of_trial,
                                                         include_switches=include_switches,)
    plot_shuff_and_true_slopes_formatted(shuff_slopes=shuff_slopes,
                              true_slopes=true_slopes,
                              metrics=metrics,
                              trial_labels=trial_labels,
                              is_first_seg_of_trial=is_first_seg_of_trial,
                              confidence_level=confidence_level,
                              include_switches=include_switches,
                               subject_id = subject_id,
                           n_shifts=n_shifts,
                              save_fig=save_fig,
                             offset=offset,
                             despine=despine,
                                    legendon=legendon,
                                    titleon=titleon)
    print(subject_id)
    print('is_first_seg_of_trial: ', is_first_seg_of_trial)
    compare_shuff_true_slopes_p(shuff_slopes, true_slopes)